# 🧮 Masterclass 12: Instance-Based & Probabilistic Classifiers
This notebook details lazy instance learners and conditional Bayesian networks:

1. **Project 1 (Scratch)**: A from-scratch `KNearestNeighbors` Euclidean distance classifier and standard `GaussianNaiveBayes` probability calculator.
2. **Project 2 (Applied)**: An email spam filtering NLP text classification pipeline using CountVectorizer + Multinomial Naive Bayes.


## 📐 Part 1: Mathematical Foundations
KNN assigns labels based on majority votes within Euclidean distance spheres:
$$d(p, q) = \sqrt{\sum_{i=1}^{n} (p_i - q_i)^2}$$
Naive Bayes utilizes Bayes' rule assuming conditional feature independence:
$$P(C_k | x) = \frac{P(C_k) \prod P(x_i | C_k)}{P(x)}$$


In [ ]:
import numpy as np

class KNNClassifierScratch:
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        preds = []
        for x in X:
            dists = np.linalg.norm(self.X_train - x, axis=1)
            nearest = np.argsort(dists)[:self.k]
            labels = self.y_train[nearest]
            preds.append(np.argmax(np.bincount(labels)))
        return np.array(preds)

class GaussianNaiveBayesScratch:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.mean = np.array([X[y == c].mean(axis=0) for c in self.classes])
        self.var = np.array([X[y == c].var(axis=0) for c in self.classes])
        self.priors = np.array([len(X[y == c]) / len(X) for c in self.classes])

    def _pdf(self, class_idx, x):
        mean = self.mean[class_idx]
        var = self.var[class_idx] + 1e-9
        numerator = np.exp(-((x - mean) ** 2) / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator

    def predict(self, X):
        preds = []
        for x in X:
            posteriors = []
            for idx, c in enumerate(self.classes):
                prior = np.log(self.priors[idx])
                conditional = np.sum(np.log(self._pdf(idx, x)))
                posteriors.append(prior + conditional)
            preds.append(self.classes[np.argmax(posteriors)])
        return np.array(preds)


## 🧪 Project 2: Multinomial Spam Classifier


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

texts = ['Urgent! Claim reward now!', 'Hey, are we still meeting today?', 'Free lottery tickets claim here', 'Can you review the report?']
labels = [1, 0, 1, 0] # 1 = Spam, 0 = Ham

spam_pipeline = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('classifier', MultinomialNB())
])
spam_pipeline.fit(texts, labels)
print('Prediction on test text:', spam_pipeline.predict(['Claim free rewards today!']))
